# Временные метрики бокового сигнала

Ноутбук описывает время начала отклонения и время экстремума подписанных
ансамблей `33.03` относительно принятого R-зубца. Он не называет эти точки
открытием клапана, механической систолой или пиком кровенаполнения без
независимого временного референса.


## Определения

Время экстремума — положение максимального по модулю отклонения среднего
ансамбля после R. Время начала — первое пересечение 10 % амплитуды того же
знакового экстремума. Методическая устойчивость оценивается повторением для
порогов 5, 10 и 20 %. Эти величины являются производными описательными
метриками и зависят от фильтрации, базового интервала, временной сетки и
групповой задержки измерительного тракта.

Если групповая задержка не измерена и не имеет статуса `accepted`, сохраняются
только нескорректированные времена со статусом
`descriptive_uncorrected_timing`.


In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path

import numpy as np

REAL_MODE = os.environ.get("KALMYKOV_RUN_REAL", "0") == "1"


def temporal_metrics(time_s, waveform, thresholds=(0.05, 0.10, 0.20)):
    time_s = np.asarray(time_s, dtype=float)
    waveform = np.asarray(waveform, dtype=float)
    post = np.flatnonzero(time_s >= 0.0)
    if len(post) < 2 or len(time_s) != len(waveform):
        raise ValueError("Нужен ансамбль с временной осью после R")
    extremum_index = int(post[np.argmax(np.abs(waveform[post]))])
    extremum_value = float(waveform[extremum_index])
    if extremum_value == 0:
        raise RuntimeError("Нулевой ансамбль не имеет определимого экстремума")
    onset = {}
    sign = np.sign(extremum_value)
    for fraction in thresholds:
        target = abs(extremum_value) * float(fraction)
        candidates = post[(sign * waveform[post] >= target) & (post <= extremum_index)]
        onset[str(fraction)] = float(time_s[candidates[0]]) if len(candidates) else None
    available = [value for value in onset.values() if value is not None]
    return {
        "extremum_time_from_r_s": float(time_s[extremum_index]),
        "extremum_value_ohm": extremum_value,
        "onset_time_from_r_s_by_fraction": onset,
        "onset_method_spread_s": float(max(available) - min(available)) if len(available) >= 2 else None,
    }


In [ ]:
time_test = np.arange(-0.15, 0.701, 0.005)
wave_test = -0.020 * np.exp(-((time_test - 0.30) / 0.08) ** 2)
metric_test = temporal_metrics(time_test, wave_test)
assert abs(metric_test["extremum_time_from_r_s"] - 0.30) <= 0.005
assert metric_test["onset_time_from_r_s_by_fraction"]["0.1"] < metric_test["extremum_time_from_r_s"]
print("33.05 synthetic_self_test: passed")


In [ ]:
if not REAL_MODE:
    print("33.05 real_data_status: blocked_until_33.03_external_artifact_exists")
else:
    config_value = os.environ.get("KALMYKOV_EXP02_CONFIG")
    if not config_value:
        raise RuntimeError("Задайте KALMYKOV_EXP02_CONFIG")
    config = json.loads(Path(config_value).expanduser().resolve().read_text(encoding="utf-8"))
    derived_root = Path(config["derived_root"]).expanduser().resolve()
    analysis_dir = derived_root / "exp02" / "analysis"
    ensembles = json.loads((analysis_dir / "33.03_ensembles.json").read_text(encoding="utf-8"))
    if ensembles.get("status") != "accepted_input_conditional_ensembles":
        raise RuntimeError("Неподходящий статус 33.03")
    operator = ensembles.get("signal_operator", {})
    timing_accepted = operator.get("timing_calibration_status") == "accepted"
    group_delay_s = float(operator.get("group_delay_s") or 0.0)
    results = []
    for item in ensembles["ensembles"]:
        metric = temporal_metrics(item["time_from_r_s"], item["mean_ohm"])
        corrected = None
        corrected_onsets = None
        if timing_accepted:
            corrected = metric["extremum_time_from_r_s"] - group_delay_s
            corrected_onsets = {
                key: (None if value is None else value - group_delay_s)
                for key, value in metric["onset_time_from_r_s_by_fraction"].items()
            }
        results.append({
            "record_id": item["record_id"],
            "subject_id": item["subject_id"],
            "size_mm": item["size_mm"],
            "mode": item["mode"],
            **metric,
            "corrected_extremum_time_from_r_s": corrected,
            "corrected_onset_time_from_r_s_by_fraction": corrected_onsets,
        })
    artifact = {
        "schema_version": 1,
        "analysis": "33.05_temporal_metrics",
        "status": "descriptive_timing" if timing_accepted else "descriptive_uncorrected_timing",
        "timing_calibration_status": operator.get("timing_calibration_status"),
        "group_delay_s": operator.get("group_delay_s"),
        "prohibited_interpretations_without_external_reference": [
            "valve_event", "mechanical_systole", "regional_source_localization",
        ],
        "results": results,
    }
    out_path = analysis_dir / "33.05_temporal_metrics.json"
    out_path.write_text(json.dumps(artifact, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    print("33.05 real_data_status: descriptive_result_written", out_path)


## Граница результата

R-зубец задаёт электрическую временную привязку. Даже после коррекции групповой
задержки он не определяет анатомическое место возникновения сигнала.
Расчётное окно $QT=QT_c\sqrt{RR}$ из `11.02` является модельным окном, а не
измеренной QT-разметкой; оно не используется здесь как независимый референс.
Для физиологического названия временной точки требуется синхронная
эхокардиография, фонокардиография или другой заранее выбранный метод.
